# Window functions: ranking, LAG and running totals

A window function computes a value for each row from a group of related rows (its window) without collapsing them. PARTITION BY defines the groups, ORDER BY the order inside each group. ROW_NUMBER, RANK and DENSE_RANK rank rows, LAG and LEAD read neighbouring rows, and sum() OVER builds running totals.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [Window functions: ranking, LAG and running totals](https://www.datapreplab.com/hub#topic/sql-window-functions) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** rank, compare with the previous row, build running totals and deduplicate.

**Part A: data.**

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  ('Asha', DATE'2026-01-01', 100), ('Asha', DATE'2026-01-05', 50), ('Asha', DATE'2026-01-09', 30),
  ('Ben',  DATE'2026-01-02', 80),  ('Ben',  DATE'2026-01-07', 20)
AS t(customer, order_date, amount);

**Part B: number, previous and running total.** Predict Asha's running total on 5 January, then run:

In [ ]:
%sql
SELECT customer, order_date, amount,
       row_number() OVER w AS order_no,
       lag(amount) OVER w AS previous_amount,
       sum(amount) OVER (PARTITION BY customer ORDER BY order_date
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM orders
WINDOW w AS (PARTITION BY customer ORDER BY order_date)
ORDER BY customer, order_date;

**Part C: grand total versus running total.**

In [ ]:
%sql
SELECT customer, order_date, amount,
       sum(amount) OVER (PARTITION BY customer) AS customer_total,
       round(amount / sum(amount) OVER (PARTITION BY customer), 2) AS share
FROM orders ORDER BY customer, order_date;

**Part D: rank the ties.**

In [ ]:
%sql
SELECT salary, row_number() OVER (ORDER BY salary DESC) AS rn,
       rank() OVER (ORDER BY salary DESC) AS rk, dense_rank() OVER (ORDER BY salary DESC) AS drk
FROM VALUES (90), (80), (80), (70) AS t(salary);

**Part E: latest order per customer.**

In [ ]:
%sql
SELECT * FROM orders
QUALIFY row_number() OVER (PARTITION BY customer ORDER BY order_date DESC) = 1;

Expected evidence:

```text
Part B -> Asha 1 NULL 100 ; 2 100 150 ; 3 50 180 ; Ben 1 NULL 80 ; 2 80 100
Part C -> Asha total 180 on each row (shares 0.56, 0.28, 0.17) ; Ben 100 (0.8, 0.2)
Part D -> rn 1 2 3 4 ; rk 1 2 2 4 ; drk 1 2 2 3
Part E -> Asha 2026-01-09 30 ; Ben 2026-01-07 20
```